<a href="https://colab.research.google.com/github/lswoodentoys/research_paper/blob/main/So_s%C3%A1nh_t%C6%B0%C6%A1ng_quan_d%E1%BA%A3i_ph%E1%BB%95_%C4%91%C3%A8n_Xenon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#BẢNG TỔNG HỢP FULL DÃY ĐÈN XENON_ISO 4892-2

In [ ]:
# ============================================================
# UV SPECTRAL SIMILARITY ANALYSIS
# G177 vs UVA-340
#
# - Exact custom R formula
# - Pearson correlation
# - Spearman correlation
# - R²
# - 20 wavelength ranges
# - Threshold sensitivity: 0.90 / 0.85 / 0.80
# - Threshold Sensitivity Matrix
# - Selection Matrix
# - Excel export
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from IPython.display import display
from google.colab import files


# ============================================================
# 2. FILE PATH
# ============================================================

file_path = (
    "/content/drive/MyDrive/"
    "Python_study/datasets/spectral_xenon_data.csv"
)


# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(file_path)

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()

print("=" * 80)
print("DATA LOADED")
print("=" * 80)

print(f"Rows before cleaning : {len(df)}")
print(f"Columns              : {list(df.columns)}")


# ============================================================
# 4. SELECT REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Wavelength_nm",
    "I_G177",
    "I_UVA340"
]

df = df[required_columns].copy()


# ============================================================
# 5. CONVERT TO NUMERIC
# ============================================================

for col in required_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ============================================================
# 6. CLEAN DATA
# ============================================================

df = df.dropna()

# G177 must be > 0 because it is used
# as denominator in the R formula
df = df[df["I_G177"] > 0]

# Sort by wavelength
df = df.sort_values(
    "Wavelength_nm"
)

# If duplicated wavelengths exist,
# calculate their mean
df = (
    df.groupby(
        "Wavelength_nm",
        as_index=False
    )[["I_G177", "I_UVA340"]]
    .mean()
)


print("\nAfter cleaning:")
print(f"Rows : {len(df)}")
print(
    f"Wavelength range : "
    f"{df['Wavelength_nm'].min()} - "
    f"{df['Wavelength_nm'].max()} nm"
)


# ============================================================
# 7. DEFINE WAVELENGTH RANGES
# ============================================================

ranges = [

    # 295-based ranges
    (295, 330),
    (295, 335),
    (295, 340),
    (295, 345),
    (295, 350),

    # 320-based ranges
    (320, 330),
    (320, 335),
    (320, 340),
    (320, 345),
    (320, 350),

    # 330-based ranges
    (330, 340),
    (330, 345),
    (330, 350),
    (330, 355),
    (330, 360),

    # 335-based ranges
    (335, 345),
    (335, 350),
    (335, 355),

    # 340-based ranges
    (340, 350),
    (340, 360)
]


# ============================================================
# 8. DEFINE THRESHOLDS
# ============================================================

thresholds = [
    0.90,
    0.85,
    0.80
]


# ============================================================
# 9. FUNCTION:
#    CALCULATE SPECTRAL METRICS
# ============================================================

def calculate_metrics(
    data,
    lambda_min,
    lambda_max
):

    # --------------------------------------------------------
    # Select wavelength range
    # --------------------------------------------------------

    subset = data[
        (data["Wavelength_nm"] >= lambda_min) &
        (data["Wavelength_nm"] <= lambda_max)
    ].copy()

    # Need at least 2 points
    if len(subset) < 2:
        return None


    # --------------------------------------------------------
    # EXACT R FORMULA
    #
    # R = 1 - sqrt(
    #       sum[
    #       (I_G177 - I_UVA340)^2 / I_G177
    #       ]
    #       /
    #       (x - 295)
    #     )
    #
    # For a general range:
    #
    # denominator = lambda_max - lambda_min
    # --------------------------------------------------------

    difference = (
        subset["I_G177"]
        -
        subset["I_UVA340"]
    )


    squared_difference = (
        difference ** 2
    )


    chi_square_term = (
        squared_difference
        /
        subset["I_G177"]
    )


    chi_square_sum = (
        chi_square_term.sum()
    )


    # IMPORTANT:
    # This follows your formula exactly.
    # It uses wavelength span, NOT number of points.

    denominator = (
        lambda_max
        -
        lambda_min
    )


    if denominator <= 0:
        return None


    SD_custom = np.sqrt(
        chi_square_sum
        /
        denominator
    )


    R = (
        1
        -
        SD_custom
    )


    # --------------------------------------------------------
    # PEARSON CORRELATION
    # --------------------------------------------------------

    pearson_r, pearson_p = pearsonr(
        subset["I_G177"],
        subset["I_UVA340"]
    )


    # --------------------------------------------------------
    # SPEARMAN CORRELATION
    # --------------------------------------------------------

    spearman_rho, spearman_p = spearmanr(
        subset["I_G177"],
        subset["I_UVA340"]
    )


    # --------------------------------------------------------
    # R-SQUARED
    #
    # For standard Pearson correlation:
    # R² = Pearson r²
    # --------------------------------------------------------

    R_squared = (
        pearson_r ** 2
    )


    # --------------------------------------------------------
    # DOES RANGE CONTAIN 340 nm?
    # --------------------------------------------------------

    contains_340 = (
        lambda_min <= 340 <= lambda_max
    )


    # --------------------------------------------------------
    # RETURN RESULTS
    # --------------------------------------------------------

    return {

        "Range_nm":
            f"{lambda_min}-{lambda_max}",

        "Lambda_min":
            lambda_min,

        "Lambda_max":
            lambda_max,

        "N":
            len(subset),

        "Chi_Square_Sum":
            chi_square_sum,

        "SD_custom":
            SD_custom,

        "R":
            R,

        "Pearson":
            pearson_r,

        "Pearson_p":
            pearson_p,

        "Spearman":
            spearman_rho,

        "Spearman_p":
            spearman_p,

        "R_squared":
            R_squared,

        "Contains_340nm":
            contains_340
    }


# ============================================================
# 10. CALCULATE BASE RESULTS
# ============================================================

base_results = []


for lambda_min, lambda_max in ranges:

    result = calculate_metrics(
        df,
        lambda_min,
        lambda_max
    )

    if result is not None:

        base_results.append(
            result
        )


results_df = pd.DataFrame(
    base_results
)


# ============================================================
# 11. SORT RESULTS
# ============================================================

results_df = (
    results_df
    .reset_index(drop=True)
)


# ============================================================
# 12. DISPLAY BASE RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("BASE SPECTRAL ANALYSIS")
print("=" * 80)

display(
    results_df.style
    .format({
        "Chi_Square_Sum": "{:.6f}",
        "SD_custom": "{:.6f}",
        "R": "{:.6f}",
        "Pearson": "{:.6f}",
        "Pearson_p": "{:.6e}",
        "Spearman": "{:.6f}",
        "Spearman_p": "{:.6e}",
        "R_squared": "{:.6f}"
    })
)


# ============================================================
# 13. RUN ALL THRESHOLDS
# ============================================================

all_threshold_results = []


for threshold in thresholds:

    temp = results_df.copy()


    # --------------------------------------------------------
    # PASS / FAIL
    # --------------------------------------------------------

    temp["R_PASS"] = (
        temp["R"] > threshold
    )


    temp["Pearson_PASS"] = (
        temp["Pearson"] > threshold
    )


    temp["R2_PASS"] = (
        temp["R_squared"] > threshold
    )


    temp["Spearman_PASS"] = (
        temp["Spearman"] > threshold
    )


    # --------------------------------------------------------
    # COUNT CONDITIONS
    # --------------------------------------------------------

    temp["Conditions_Passed"] = (

        temp["R_PASS"].astype(int)

        +

        temp["Pearson_PASS"].astype(int)

        +

        temp["R2_PASS"].astype(int)

        +

        temp["Spearman_PASS"].astype(int)

    )


    # --------------------------------------------------------
    # ALL FOUR CONDITIONS
    # --------------------------------------------------------

    temp["All_4_Conditions"] = (
        temp["Conditions_Passed"] == 4
    )


    # --------------------------------------------------------
    # FINAL SELECTION
    #
    # SELECTED =
    #   1. Contains 340 nm
    #   2. All 4 conditions pass
    # --------------------------------------------------------

    temp["Selection"] = np.where(

        temp["Contains_340nm"] &
        temp["All_4_Conditions"],

        "SELECTED",

        np.where(

            temp["Contains_340nm"],

            "340_INCLUDED",

            "NOT_RELEVANT"
        )
    )


    # Add threshold
    temp["Threshold"] = threshold


    # Store
    all_threshold_results.append(
        temp
    )


# ============================================================
# 14. COMBINE ALL THRESHOLDS
# ============================================================

sensitivity_detail = pd.concat(
    all_threshold_results,
    ignore_index=True
)


# ============================================================
# 15. CREATE THRESHOLD SENSITIVITY MATRIX
# ============================================================

matrix_rows = []


for range_name in results_df["Range_nm"]:

    row = {
        "Range_nm":
            range_name
    }


    # Check each threshold
    for threshold in thresholds:

        temp = sensitivity_detail[
            (sensitivity_detail["Range_nm"] == range_name)
            &
            (sensitivity_detail["Threshold"] == threshold)
        ]


        if len(temp) == 0:
            continue


        temp = temp.iloc[0]


        conditions = int(
            temp["Conditions_Passed"]
        )


        if temp["All_4_Conditions"]:

            status = "4/4"

        else:

            status = (
                f"{conditions}/4"
            )


        row[
            f"{threshold:.2f}"
        ] = status


    matrix_rows.append(
        row
    )


threshold_matrix = pd.DataFrame(
    matrix_rows
)


# ============================================================
# 16. ADD 340 nm INFORMATION
# ============================================================

contains_340_df = results_df[
    [
        "Range_nm",
        "Contains_340nm"
    ]
].copy()


threshold_matrix = threshold_matrix.merge(
    contains_340_df,
    on="Range_nm",
    how="left"
)


# Reorder
threshold_matrix = threshold_matrix[
    [
        "Range_nm",
        "Contains_340nm",
        "0.90",
        "0.85",
        "0.80"
    ]
]


# ============================================================
# 17. DISPLAY THRESHOLD SENSITIVITY MATRIX
# ============================================================

print("\n")
print("=" * 80)
print("THRESHOLD SENSITIVITY MATRIX")
print("=" * 80)

display(
    threshold_matrix
)


# ============================================================
# 18. CREATE FINAL SELECTION MATRIX
# ============================================================

selection_matrix = sensitivity_detail[
    [
        "Range_nm",
        "Threshold",
        "Contains_340nm",

        "R",
        "Pearson",
        "R_squared",
        "Spearman",

        "R_PASS",
        "Pearson_PASS",
        "R2_PASS",
        "Spearman_PASS",

        "Conditions_Passed",
        "All_4_Conditions",
        "Selection"
    ]
].copy()


selection_matrix = (
    selection_matrix
    .sort_values(
        [
            "Range_nm",
            "Threshold"
        ]
    )
    .reset_index(drop=True)
)


# ============================================================
# 19. DISPLAY FINAL SELECTION MATRIX
# ============================================================

print("\n")
print("=" * 80)
print("DETAILED SELECTION MATRIX")
print("=" * 80)

display(
    selection_matrix.style
    .format({
        "R": "{:.6f}",
        "Pearson": "{:.6f}",
        "R_squared": "{:.6f}",
        "Spearman": "{:.6f}"
    })
)


# ============================================================
# 20. SHOW SELECTED RANGES FOR EACH THRESHOLD
# ============================================================

print("\n")
print("=" * 80)
print("SELECTED RANGES BY THRESHOLD")
print("=" * 80)


for threshold in thresholds:

    selected = selection_matrix[
        (selection_matrix["Threshold"] == threshold)
        &
        (selection_matrix["Selection"] == "SELECTED")
    ]


    print(
        f"\nThreshold = {threshold:.2f}"
    )


    if len(selected) == 0:

        print(
            "No range satisfies all 4 conditions "
            "and includes 340 nm."
        )

    else:

        display(
            selected[
                [
                    "Range_nm",
                    "R",
                    "Pearson",
                    "R_squared",
                    "Spearman",
                    "Conditions_Passed"
                ]
            ]
            .style
            .format({
                "R": "{:.6f}",
                "Pearson": "{:.6f}",
                "R_squared": "{:.6f}",
                "Spearman": "{:.6f}"
            })
        )


# ============================================================
# 21. NUMERIC MATRIX FOR HEATMAP
# ============================================================

heatmap_data = []


for range_name in results_df["Range_nm"]:

    row = []


    for threshold in thresholds:

        temp = sensitivity_detail[
            (sensitivity_detail["Range_nm"] == range_name)
            &
            (sensitivity_detail["Threshold"] == threshold)
        ]


        value = int(
            temp["Conditions_Passed"].iloc[0]
        )


        row.append(
            value
        )


    heatmap_data.append(
        row
    )


heatmap_df = pd.DataFrame(

    heatmap_data,

    index=results_df["Range_nm"],

    columns=[
        f"{x:.2f}"
        for x in thresholds
    ]
)


# ============================================================
# 22. HEATMAP: CONDITIONS PASSED
# ============================================================

plt.figure(
    figsize=(10, 10)
)


plt.imshow(
    heatmap_df.values,
    aspect="auto"
)


plt.xticks(
    range(len(thresholds)),
    [
        f"Threshold {x:.2f}"
        for x in thresholds
    ]
)


plt.yticks(
    range(len(heatmap_df.index)),
    heatmap_df.index
)


plt.xlabel(
    "Threshold"
)


plt.ylabel(
    "Wavelength Range"
)


plt.title(
    "Threshold Sensitivity Matrix\n"
    "Number of Conditions Passed (0–4)"
)


# Cell labels
for i in range(
    len(heatmap_df.index)
):

    for j in range(
        len(heatmap_df.columns)
    ):

        value = heatmap_df.iloc[i, j]

        plt.text(
            j,
            i,
            str(value),
            ha="center",
            va="center"
        )


plt.colorbar(
    label="Conditions Passed"
)


plt.tight_layout()

plt.show()


# ============================================================
# 23. SELECTED / NOT SELECTED MATRIX
# ============================================================

selection_status = []


for range_name in results_df["Range_nm"]:

    row = []


    for threshold in thresholds:

        temp = sensitivity_detail[
            (sensitivity_detail["Range_nm"] == range_name)
            &
            (sensitivity_detail["Threshold"] == threshold)
        ]


        selection = (
            temp["Selection"].iloc[0]
        )


        if selection == "SELECTED":

            value = 1

        else:

            value = 0


        row.append(
            value
        )


    selection_status.append(
        row
    )


selection_status_df = pd.DataFrame(

    selection_status,

    index=results_df["Range_nm"],

    columns=[
        f"{x:.2f}"
        for x in thresholds
    ]
)


# ============================================================
# 24. HEATMAP: SELECTED RANGE
# ============================================================

plt.figure(
    figsize=(8, 10)
)


plt.imshow(
    selection_status_df.values,
    aspect="auto"
)


plt.xticks(
    range(len(thresholds)),
    [
        f"{x:.2f}"
        for x in thresholds
    ]
)


plt.yticks(
    range(len(selection_status_df.index)),
    selection_status_df.index
)


plt.xlabel(
    "Threshold"
)


plt.ylabel(
    "Wavelength Range"
)


plt.title(
    "Selected Range Sensitivity\n"
    "1 = All 4 Conditions + 340 nm"
)


# Cell labels
for i in range(
    len(selection_status_df.index)
):

    for j in range(
        len(selection_status_df.columns)
    ):

        value = (
            selection_status_df
            .iloc[i, j]
        )


        text_value = (
            "YES"
            if value == 1
            else "NO"
        )


        plt.text(
            j,
            i,
            text_value,
            ha="center",
            va="center"
        )


plt.tight_layout()

plt.show()


# ============================================================
# 25. EXPORT EVERYTHING TO ONE EXCEL FILE
# ============================================================

output_excel = (
    "/content/drive/MyDrive/"
    "Threshold_Sensitivity_Analysis.xlsx"
)


with pd.ExcelWriter(
    output_excel,
    engine="openpyxl"
) as writer:

    # Sheet 1
    results_df.to_excel(
        writer,
        sheet_name="Base_Metrics",
        index=False
    )


    # Sheet 2
    sensitivity_detail.to_excel(
        writer,
        sheet_name="Threshold_Detail",
        index=False
    )


    # Sheet 3
    threshold_matrix.to_excel(
        writer,
        sheet_name="Threshold_Sensitivity",
        index=False
    )


    # Sheet 4
    selection_matrix.to_excel(
        writer,
        sheet_name="Selection_Matrix",
        index=False
    )


    # Sheet 5
    heatmap_df.to_excel(
        writer,
        sheet_name="Conditions_Heatmap"
    )


    # Sheet 6
    selection_status_df.to_excel(
        writer,
        sheet_name="Selected_Heatmap"
    )


# ============================================================
# 26. EXPORT THRESHOLD MATRIX TO CSV
# ============================================================

output_csv = (
    "/content/drive/MyDrive/"
    "Threshold_Sensitivity_Matrix.csv"
)


threshold_matrix.to_csv(
    output_csv,
    index=False
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("ANALYSIS COMPLETED")
print("=" * 80)

print(
    f"Number of wavelength ranges : "
    f"{len(results_df)}"
)

print(
    f"Thresholds tested           : "
    f"{thresholds}"
)

print(
    f"Excel file                  : "
    f"{output_excel}"
)

print(
    f"CSV file                    : "
    f"{output_csv}"
)


# ============================================================
# 28. FINAL SELECTED-RANGE SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("FINAL THRESHOLD SUMMARY")
print("=" * 80)


summary_rows = []


for threshold in thresholds:

    selected = selection_matrix[
        (selection_matrix["Threshold"] == threshold)
        &
        (selection_matrix["Selection"] == "SELECTED")
    ]


    if len(selected) > 0:

        ranges_selected = ", ".join(
            selected["Range_nm"].tolist()
        )

    else:

        ranges_selected = "None"


    summary_rows.append({

        "Threshold":
            threshold,

        "Selected_Ranges":
            ranges_selected,

        "Number_Selected":
            len(selected)
    })


final_summary = pd.DataFrame(
    summary_rows
)


display(
    final_summary
)


# ============================================================
# 29. DOWNLOAD EXCEL
# ============================================================

files.download(
    output_excel
)

print("\nExcel download started.")